# GNN : Graph Neural Networks en Chimie avec DeepChem
Faire un GNN "à la main" avec Keras est un cauchemar mathématique (car chaque molécule a un nombre d'atomes différent, ce que les réseaux classiques détestent).

Heureusement, la communauté scientifique a créé **DeepChem**, une bibliothèque open-source qui rend la création de GNN incroyablement simple !

In [19]:
# 1. Installation de DeepChem (retirer le # pour exécuter)
#%pip install deepchem
# Note : DeepChem peut prendre un peu de temps à s'installer.
#%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu --upgrade


In [20]:
# --- CODE POUR MASQUER LES AVERTISSEMENTS ---
import logging
# On demande au journal de DeepChem de se taire sauf en cas d'erreur fatale
logging.getLogger("deepchem").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore') # Masque les avertissements Python standards
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')    # Masque les avertissements rouges de RDKit
# --------------------------------------------

import deepchem as dc
import numpy as np


# 2. LE "FEATURIZER" (L'extracteur de caractéristiques)
# Au lieu de transformer la molécule en une liste de 0 et 1 (Morgan Fingerprint),
# ConvMolFeaturizer transforme la molécule en un VRAI GRAPHE (noeuds = atomes, arêtes = liaisons chimiques).
featurizer = dc.feat.ConvMolFeaturizer()

# 3. CHARGEMENT AUTOMATIQUE DES DONNÉES
# DeepChem est magique : il a le dataset ClinTox intégré !
# Il le télécharge, le nettoie, applique le Featurizer (graphe) et sépare l'entraînement et le test tout seul.
print("Préparation des graphes moléculaires...")
tasks, datasets, transformers = dc.molnet.load_clintox(featurizer=featurizer)
train_dataset, valid_dataset, test_dataset = datasets

print("\nLes tâches à prédire :", tasks)
print(f"Taille Entraînement: {len(train_dataset)}")
print(f"Taille Test: {len(test_dataset)}")

Préparation des graphes moléculaires...

Les tâches à prédire : ['FDA_APPROVED', 'CT_TOX']
Taille Entraînement: 1184
Taille Test: 148


In [21]:
# 4. CRÉATION DU RÉSEAU DE NEURONES EN GRAPHE (GNN)
# GraphConvModel est l'architecture GNN standard en chimie.
# Il va "lire" les graphes, atome par atome, et comprendre la structure 3D/2D.
model = dc.models.GraphConvModel(
    n_tasks=len(tasks), 
    mode='classification',
    dropout=0.2 # Toujours un peu de sécurité contre le surapprentissage
)

# 5. ENTRAÎNEMENT
print("\nEntraînement du GNN en cours (c'est souvent plus rapide qu'un réseau classique !)...")
model.fit(train_dataset, nb_epoch=20)
print("Entraînement terminé !")


Entraînement du GNN en cours (c'est souvent plus rapide qu'un réseau classique !)...
Entraînement terminé !


In [22]:
# 6. ÉVALUATION DU MODÈLE
# On utilise la métrique ROC-AUC (très utilisée en médecine pour évaluer les faux positifs/négatifs)
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)

train_scores = model.evaluate(train_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

print(f"\nScore sur les données d'entraînement : {train_scores['roc_auc_score']:.3f}")
print(f"Score sur les données de test : {test_scores['roc_auc_score']:.3f}")


Score sur les données d'entraînement : 0.950
Score sur les données de test : 0.827


In [23]:
# 7. TEST SUR VOS PROPRES MOLÉCULES
# On va retester le fameux Valdécoxib (qui avait trompé notre IA classique)

smiles_a_tester = [
    "CC(=O)NC1=CC=C(O)C=C1",                             # Paracétamol (Doliprane) - Sûr
    "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N" # Valdécoxib (Bextra) - TOXIQUE ! L'IA classique avait dit sûr à 99%
]

# On convertit nos textes en Graphes avec le même featurizer
graphes = featurizer.featurize(smiles_a_tester)

# On crée un petit dataset "à la volée" pour DeepChem
dataset_perso = dc.data.NumpyDataset(X=graphes)

# Prédiction ! 
# Attention, DeepChem sort des prédictions pour les 2 tâches de ClinTox (FDA_APPROVED et FDA_TOX).
# L'indice [:, 1, 1] permet de récupérer la probabilité d'être Toxique (Tâche 2, Classe 1).
predictions = model.predict(dataset_perso)[:, 1, 1]

print("\n--- RÉSULTATS DU GNN ---")
print(f"Paracétamol : {predictions[0]*100:.2f}% de risque de toxicité")
print(f"Valdécoxib  : {predictions[1]*100:.2f}% de risque de toxicité")


--- RÉSULTATS DU GNN ---
Paracétamol : 75.69% de risque de toxicité
Valdécoxib  : 73.90% de risque de toxicité
